<a href="https://colab.research.google.com/github/dhan4567/Machine_Learning_Projects/blob/main/Restaurant_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load **Libraries**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder

Load Dataset

In [ ]:
df = pd.read_csv('/content/Dataset .csv')

In [ ]:
print(df.head())
print(df.shape)
print(df.info())

   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, Mandaluyong City   
3      SM 

Handling Missing **Values**

In [ ]:
df.drop_duplicates(inplace=True)

df['Cuisines']=df['Cuisines'].fillna("Unknown")
df['Price range']=df['Price range'].fillna(
    df['Price range'].mode()[0]
)
df['Aggregate rating']=df['Aggregate rating'].fillna(
    df['Aggregate rating'].mean()
)


/tmp/ipykernel_3507/3619989630.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)
/tmp/ipykernel_3507/3619989630.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Cuisines']=df['Cuisines'].fillna("Unknown")
/tmp/ipykernel_3507/3619989630.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-co

Encode Categorical **Variables**

In [ ]:
encoder = LabelEncoder()
df['Price_range_encoded'] = encoder.fit_transform(df['Price range'])

In [ ]:
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Locality Verbose_encoded,Cuisines_encoded,Currency_encoded,Has Table booking_encoded,Has Online delivery_encoded,Is delivering now_encoded,Switch to order menu_encoded,Rating color_encoded,Rating text_encoded,content_encoded
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,148,717,0,1,0,0,0,0,0,890
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,528,882,0,1,0,0,0,0,0,1094
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,272,1240,0,1,0,0,0,1,2,1587
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,749,894,0,0,0,0,0,0,0,1119
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,749,890,0,1,0,0,0,0,0,1108


Select Recommendation **Criteria**

In [ ]:
df['content']=(
    df['Cuisines'].astype(str)+" "+
    df['Price range'].astype(str)+" "+
    df['Aggregate rating'].astype(str)
)

/tmp/ipykernel_3507/2859194808.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['content']=(


Convert Text Features Into Numerical **Form**

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])

Calculate **Similarity**

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)

Build Recommendation Function

In [ ]:
def recommend_restaurants(cuisine,price_range,top_n=5):

  user_input = cuisine+ " "+str(price_range)
  user_vector = tfidf.transform([user_input])

  similarity_scores = cosine_similarity(
      user_vector,tfidf_matrix
  )
  scores = similarity_scores.flatten()

  top_indices = scores.argsort()[-top_n:][::-1]

  recommendations = df.iloc[top_indices][
     ['Restaurant Name',
      'Cuisines',
      'Price range',
      'Aggregate rating']

  ]
  return recommendations

Test Recommend **System**

In [ ]:
recommend_restaurants(
    cuisine="Japanese",
    price_range=3
)

,Restaurant Name,Cuisines,Price range,Aggregate rating
429,Marukame Udon,Japanese,1,4.9
1,Izakaya Kikufuji,Japanese,3,4.5
1466,Kuuraku,Japanese,3,3.9
54,Sushi Leblon,Japanese,4,4.6
222,Soho Hibachi,Japanese,1,4.3


In [ ]:
recommend_restaurants(
    cuisine= 'Italian',
    price_range= 2
)

,Restaurant Name,Cuisines,Price range,Aggregate rating
9316,Baduzzi,Italian,4,4.6
2070,56 Fresca,Italian,2,3.7
1523,Bella Cucina - Le Meridien Gurgaon,Italian,4,4.1
9414,San Carlo,Italian,2,4.3
3240,Moets Stone,Italian,4,3.8


In [ ]:
recommend_restaurants(
    cuisine= 'Indian',
    price_range= 1
)

,Restaurant Name,Cuisines,Price range,Aggregate rating
597,Tresind - Nassima Royal Hotel,Indian,4,4.9
9372,Mother India's Cafe,Indian,2,4.4
581,Tamba,Indian,4,4.7
586,Rasoi Ghar,Indian,2,4.3
590,Carnival By Tresind,Indian,4,4.9


Bonus Improvement(Recommendation)**bold text**

In [ ]:
df["content"]= (
    df['Cuisines'].astype(str)+" "+
    df['Price range'].astype(str)+" "+
    df['Aggregate rating'].round().astype(str)
)

/tmp/ipykernel_3507/1527761092.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content"]= (


In [ ]:
df = df[df['Aggregate rating'] >= 3.5]